# 07 — α,β-CROWN Baseline (SOTA neural-network verifier)

Runs α,β-CROWN (the state-of-the-art neural-network verifier from VNN-COMP 2021–2024)
on the same MNIST evaluation indices used by the MILP verifier (notebook 03).
α,β-CROWN combines linear bound propagation with branch-and-bound over ReLU splits;
it is **sound** and (in practice) **complete** within its timeout, and orders of
magnitude faster than the big-M MILP encoder.

**Pipeline**
1. Install α,β-CROWN (clones `Verified-Intelligence/alpha-beta-CROWN` + pip-installs `auto_LiRPA`).
2. Load the trained MLP checkpoint (`runs/mlp_mnist/model.pt` *or* `runs/mlp_ibp_trained/model.pt`).
3. Export to ONNX with input shape `(1, 784)`, opset 12, and assert PyTorch ↔ ONNX-Runtime
   parity within `1e-5` on 100 random samples (loud failure if not).
4. Generate one VNNLIB spec per evaluation sample under `specs/sample_XXXX.vnnlib`,
   each encoding the L∞ box `[clip(x0-ε, 0, 1), clip(x0+ε, 0, 1)]` and the
   property *“no class c≠y outscores y”* (UNSAT ⇒ VERIFIED).
5. Build `instances.csv` with `(onnx, vnnlib, timeout=60)` rows.
6. Invoke `abcrown.py` with a custom YAML config.
7. Parse α,β-CROWN's output into `results/abcrown_<run>.csv` with columns
   `sample_idx, true_label, status, time`. Statuses are normalised to
   `VERIFIED` / `FALSIFIED` / `TIMEOUT` / `ERROR` for cross-method consistency.

**Run twice** — once per checkpoint — to cover both the standard and IBP-trained MLPs.

In [1]:
# ── Colab bootstrap ─────────────────────────────────────────────────────────────
# Mounts Drive (Colab) or noop (local), cd's into the project folder, and
# self-generates `assets/splits/mnist_eval_1000.json` if it's not present.
# colab_bootstrap_v1
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/thesis-formal-verification")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_ROOT)
    print(f"[colab] working dir: {PROJECT_ROOT}")
except ImportError:
    print(f"[local] working dir: {Path.cwd()}")

# Self-generate the 1000-sample evaluation split if missing
import json, random
split_100  = Path("assets/splits/mnist_eval_100.json")
split_1000 = Path("assets/splits/mnist_eval_1000.json")
if not split_1000.exists():
    split_100.parent.mkdir(parents=True, exist_ok=True)
    seed = 1234
    if split_100.exists():
        idx100 = json.loads(split_100.read_text(encoding="utf-8"))["indices"]
    else:
        idx100 = random.Random(seed).sample(range(10_000), 100)
        split_100.write_text(json.dumps({"seed": seed, "indices": idx100}, indent=2) + "\n", encoding="utf-8")
    remaining = [i for i in range(10_000) if i not in set(idx100)]
    extra = random.Random(seed).sample(remaining, 900)
    indices_1000 = idx100 + extra
    assert indices_1000[:100] == idx100
    split_1000.write_text(json.dumps({"seed": seed, "indices": indices_1000}, indent=2) + "\n", encoding="utf-8")
    print(f"[bootstrap] wrote {split_1000} ({len(indices_1000)} indices)")
else:
    print(f"[bootstrap] OK {split_1000}  ({len(json.loads(split_1000.read_text(encoding='utf-8'))['indices'])} indices)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[colab] working dir: /content/drive/MyDrive/thesis-formal-verification
[bootstrap] OK assets/splits/mnist_eval_1000.json  (1000 indices)


## 1 — Install α,β-CROWN (Colab / Linux only)

α,β-CROWN's installer expects a Linux shell. On Colab, the cell below works as-is.
On Windows, run this notebook inside WSL or on Colab — α,β-CROWN's PyPi build does
not target Windows.

In [2]:
import os, subprocess, sys
from pathlib import Path

# 1) Lightweight deps used by THIS notebook
!pip install -q torch torchvision numpy pandas pyyaml tqdm onnx onnxruntime onnxscript

# 2) Clone α,β-CROWN first — we install from its bundled requirements.txt
ABCROWN_REPO = Path("/content/alpha-beta-CROWN") if Path("/content").exists() else Path.cwd() / "alpha-beta-CROWN"
if not ABCROWN_REPO.exists():
    !git clone --depth 1 https://github.com/Verified-Intelligence/alpha-beta-CROWN.git {ABCROWN_REPO}
else:
    print(f"alpha-beta-CROWN already at {ABCROWN_REPO}")

# 3) Install α,β-CROWN's runtime deps (this is the key fix vs. earlier attempts:
#    we let the project pin its own deps instead of guessing).  Use --no-deps
#    on auto_LiRPA itself because its strict torch pin clashes with Colab's
#    pre-installed torch 2.10 — the actual API surface used here works fine.
REQ = ABCROWN_REPO / "complete_verifier" / "requirements.txt"
if REQ.exists():
    !pip install -q --no-deps git+https://github.com/Verified-Intelligence/auto_LiRPA.git
    !pip install -q onnx2pytorch onnxoptimizer onnxsim bidict einops sortedcontainers appdirs ml_collections termcolor jsonpatch ninja pytest-mock pytest-order

# Verify the runtime dep chain α,β-CROWN actually walks at startup.
# If any of these fails, the next abcrown run will fail with a clear error.
for _m in ['onnx', 'onnxruntime', 'onnx2pytorch', 'onnxoptimizer', 'auto_LiRPA',
           'sortedcontainers', 'appdirs', 'ml_collections', 'termcolor', 'jsonpatch']:
    try:
        __import__(_m)
    except Exception as _e:
        print(f'[WARN] {_m} import failed: {_e}')
else:
    print(f"WARN: {REQ} not found — installing fallback dep list")
    !pip install -q --no-deps git+https://github.com/Verified-Intelligence/auto_LiRPA.git
    !pip install -q onnx2pytorch sortedcontainers appdirs ml_collections termcolor jsonpatch

ABCROWN_DIR = ABCROWN_REPO / "complete_verifier"
assert (ABCROWN_DIR / "abcrown.py").exists(), f"missing {ABCROWN_DIR / 'abcrown.py'}"
print(f"abcrown.py at: {ABCROWN_DIR / 'abcrown.py'}")


alpha-beta-CROWN already at /content/alpha-beta-CROWN
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 105.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1480: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


WARN: /content/alpha-beta-CROWN/complete_verifier/requirements.txt not found — installing fallback dep list
  Preparing metadata (setup.py) ... done
abcrown.py at: /content/alpha-beta-CROWN/complete_verifier/abcrown.py


## 2 — Library code (model, eval split, ONNX/VNNLIB writers)

In [3]:
from __future__ import annotations

import json
import os
import random
import shutil
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


# ── Eval split ─────────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class Split:
    seed: int
    indices: list


def load_split(path) -> Split:
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    return Split(seed=int(obj["seed"]), indices=[int(i) for i in obj["indices"]])


# ── Model ──────────────────────────────────────────────────────────────────────
class MnistMlp(nn.Module):
    """Identical to src/nnverify/models/mlp.py — must match the trained checkpoint."""

    def __init__(self, in_dim: int = 784, h1: int = 128, h2: int = 64, num_classes: int = 10):
        super().__init__()
        self.in_dim = int(in_dim)
        self.h1 = int(h1)
        self.h2 = int(h2)
        self.num_classes = int(num_classes)
        self.fc1 = nn.Linear(self.in_dim, self.h1)
        self.fc2 = nn.Linear(self.h1, self.h2)
        self.fc3 = nn.Linear(self.h2, self.num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        if x.ndim == 4:
            x = x.view(x.shape[0], -1)
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))

    def linear_layers(self):
        return [self.fc1, self.fc2, self.fc3]


class _MlpFlatOnly(nn.Module):
    """Pure (B, 784) → (B, 10) — no view/reshape — for clean ONNX export."""

    def __init__(self, m: MnistMlp):
        super().__init__()
        self.fc1 = m.fc1
        self.fc2 = m.fc2
        self.fc3 = m.fc3
        self.relu = m.relu

    def forward(self, x):
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))


@dataclass(frozen=True)
class CheckpointMeta:
    model_type: str
    model_kwargs: dict
    run_name: str


def load_checkpoint(path, map_location="cpu"):
    payload = torch.load(str(path), map_location=map_location)
    raw = payload["meta"]
    meta = CheckpointMeta(
        model_type=str(raw["model_type"]),
        model_kwargs=dict(raw.get("model_kwargs", {})),
        run_name=str(raw.get("run_name", "run")),
    )
    if meta.model_type != "mlp":
        raise ValueError(f"Only MLP supported here; got {meta.model_type!r}")
    model = MnistMlp(**meta.model_kwargs)
    model.load_state_dict(payload["model_state_dict"])
    return model, meta, payload.get("metrics", {})


# ── ONNX export with parity check ─────────────────────────────────────────────
def export_to_onnx(model: MnistMlp, onnx_path: Path, *, atol: float = 1e-5, seed: int = 1234) -> None:
    """Export with input shape (1, 784), opset 12. Assert torch≡onnx within ``atol``."""
    onnx_path = Path(onnx_path)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    flat = _MlpFlatOnly(model).eval()
    dummy = torch.zeros(1, 784, dtype=torch.float32)
    torch.onnx.export(
        flat,
        dummy,
        str(onnx_path),
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=12,
        do_constant_folding=True,
        dynamo=False,   # legacy TorchScript exporter (no onnxscript dep)
    )
    # Parity check
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    rng = np.random.RandomState(int(seed))
    n_check = 100
    x_np = rng.uniform(0.0, 1.0, size=(n_check, 784)).astype(np.float32)
    with torch.no_grad():
        torch_out = flat(torch.from_numpy(x_np)).cpu().numpy()
    onnx_out = sess.run(["logits"], {"input": x_np})[0]
    diff = float(np.max(np.abs(torch_out - onnx_out)))
    print(f"ONNX exported: {onnx_path}  (parity max|Δ| over {n_check} random samples = {diff:.3e})")
    assert diff < atol, (
        f"ONNX parity FAILED: max|Δ|={diff:.3e} > atol={atol:.3e}. "
        "Check that opset/precision/architecture match."
    )
    print(f"OK — torch ≡ onnx within {atol:.0e}")


# ── VNNLIB writer ─────────────────────────────────────────────────────────────
def write_vnnlib_robustness(
    path: Path,
    x0_flat: np.ndarray,        # shape (784,), in [0, 1]
    eps: float,
    y: int,
    num_inputs: int = 784,
    num_classes: int = 10,
) -> None:
    """Write a VNNLIB file encoding L∞ robustness of label ``y``.

    UNSAT under the disjunction below ⇔ network is robust at (x0, y, eps).
    """
    assert x0_flat.shape == (num_inputs,)
    lo = np.clip(x0_flat - float(eps), 0.0, 1.0).astype(np.float64)
    hi = np.clip(x0_flat + float(eps), 0.0, 1.0).astype(np.float64)

    lines: List[str] = []
    lines.append(f"; Robustness spec: label={y}, eps={eps}")
    lines.append("; UNSAT  ⇒  VERIFIED   |   SAT (with witness) ⇒  FALSIFIED")
    lines.append("")
    for i in range(num_inputs):
        lines.append(f"(declare-const X_{i} Real)")
    lines.append("")
    for c in range(num_classes):
        lines.append(f"(declare-const Y_{c} Real)")
    lines.append("")
    # Input box
    for i in range(num_inputs):
        lines.append(f"(assert (>= X_{i} {lo[i]:.10f}))")
        lines.append(f"(assert (<= X_{i} {hi[i]:.10f}))")
    lines.append("")
    # Violation:  ∃ c ≠ y :  Y_c >= Y_y
    or_clauses = [f"(>= Y_{c} Y_{y})" for c in range(num_classes) if c != y]
    lines.append("(assert (or")
    for cl in or_clauses:
        lines.append(f"  {cl}")
    lines.append("))")
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")


print("Library code loaded")

Library code loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3 — Configuration

In [4]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Pick ONE of these two checkpoints (run the cell once per checkpoint):
CKPT_PATH    = "runs/mlp_mnist/model.pt"           # standard-trained MLP (from 01)
# CKPT_PATH    = "runs/mlp_ibp_trained/model.pt"   # IBP-trained MLP (from 08)

SUBSET_PATH  = "assets/splits/mnist_eval_1000.json"  # 1st 100 indices = MILP/abcrown subset
MAX_SAMPLES  = 100      # evaluate first MAX_SAMPLES indices (matches MILP subset)
DATA_DIR     = "data"
EPS          = 0.03
TIMEOUT      = 60       # per-instance α,β-CROWN timeout (seconds)
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

WORK_DIR     = Path("abcrown_work")  # holds onnx, specs/, instances.csv, out.csv
WORK_DIR.mkdir(parents=True, exist_ok=True)

set_seed(1234)
print(f"Device: {DEVICE} | eps={EPS} | timeout={TIMEOUT}s | MAX_SAMPLES={MAX_SAMPLES}")
print(f"Working dir: {WORK_DIR.resolve()}")

Device: cuda | eps=0.03 | timeout=60s | MAX_SAMPLES=100
Working dir: /content/drive/MyDrive/thesis-formal-verification/abcrown_work


## 4 — Load checkpoint and evaluation split

In [5]:
model, meta, _ = load_checkpoint(CKPT_PATH, map_location="cpu")
model.eval()
print(f"Loaded model: {meta.model_type!r}  run={meta.run_name!r}")

split = load_split(SUBSET_PATH)
indices = split.indices[:MAX_SAMPLES]
print(f"Verifying {len(indices)} samples (first {MAX_SAMPLES} of {len(split.indices)} in {SUBSET_PATH})")

Loaded model: 'mlp'  run='mlp_mnist'
Verifying 100 samples (first 100 of 1000 in assets/splits/mnist_eval_1000.json)


## 5 — Export to ONNX (with parity assertion)

In [6]:
ONNX_PATH = WORK_DIR / f"{meta.run_name}.onnx"
export_to_onnx(model, ONNX_PATH, atol=1e-5, seed=1234)

ONNX exported: abcrown_work/mlp_mnist.onnx  (parity max|Δ| over 100 random samples = 7.629e-06)
OK — torch ≡ onnx within 1e-05


/tmp/ipykernel_17321/1138386965.py:109: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:218: DeprecationWarning: The feature will be removed. Please remove usage of this function
  setup_onnx_logging(verbose) as log_ctx,


## 6 — Generate VNNLIB specs and instances.csv

α,β-CROWN consumes a `.csv` index where each row is
`<onnx_relpath>,<vnnlib_relpath>,<timeout_seconds>`. Paths are *relative to* the
directory passed via `--root_path`.

In [7]:
# Load MNIST so we can pull each x0
tfm = transforms.ToTensor()
test_ds = datasets.MNIST(root=str(DATA_DIR), train=False, download=True, transform=tfm)

specs_dir = WORK_DIR / "specs"
specs_dir.mkdir(parents=True, exist_ok=True)

csv_rows: List[Tuple[str, str, int]] = []
labels_by_idx = {}

for sample_idx in indices:
    x_t, y_t = test_ds[int(sample_idx)]
    x0 = x_t.detach().view(-1).numpy().astype(np.float32)  # (784,)
    y = int(y_t)
    labels_by_idx[int(sample_idx)] = y
    vnnlib_path = specs_dir / f"sample_{int(sample_idx):05d}.vnnlib"
    write_vnnlib_robustness(vnnlib_path, x0_flat=x0, eps=EPS, y=y)
    csv_rows.append((
        ONNX_PATH.name,                           # relative to WORK_DIR
        f"specs/{vnnlib_path.name}",              # relative to WORK_DIR
        int(TIMEOUT),
    ))

instances_csv = WORK_DIR / "instances.csv"
instances_csv.write_text(
    "\n".join(f"{a},{b},{c}" for (a, b, c) in csv_rows) + "\n",
    encoding="utf-8",
)
print(f"Wrote {len(csv_rows)} VNNLIB specs to {specs_dir}")
print(f"Wrote instances index: {instances_csv}")
print("\nFirst 3 rows of instances.csv:")
for line in instances_csv.read_text().splitlines()[:3]:
    print("  ", line)

Wrote 100 VNNLIB specs to abcrown_work/specs
Wrote instances index: abcrown_work/instances.csv

First 3 rows of instances.csv:
   mlp_mnist.onnx,specs/sample_07220.vnnlib,60
   mlp_mnist.onnx,specs/sample_01914.vnnlib,60
   mlp_mnist.onnx,specs/sample_00122.vnnlib,60


## 7 — Run α,β-CROWN

We invoke `abcrown.py` with a tiny custom YAML config. The verifier writes a
result CSV to `abcrown_work/abcrown_out.csv` (one row per instance with
α,β-CROWN's native status strings). We post-process this into our schema.

> Output statuses produced by α,β-CROWN: `safe`, `unsafe-pgd`, `unsafe-bab`,
> `timeout`, `unknown`, `error`. Mapping to our schema:
> - `safe` → `VERIFIED`
> - `unsafe-*` → `FALSIFIED`
> - `timeout` → `TIMEOUT`
> - else → `ERROR`

In [8]:
ABCROWN_OUT = WORK_DIR / "abcrown_out.csv"
ABCROWN_LOG = WORK_DIR / "abcrown_stdout.log"
ABCROWN_CFG = WORK_DIR / "abcrown_mnist_mlp.yaml"

abcrown_cfg_text = f"""\
general:
  device: {DEVICE}
  csv_name: instances.csv
  results_file: abcrown_out.csv
  root_path: {WORK_DIR.resolve()}
  conv_mode: matrix
  enable_incomplete_verification: true
model:
  onnx_path: null   # taken per-instance from instances.csv
specification:
  norm: .inf
  type: lp
  vnnlib_path: null # taken per-instance from instances.csv
attack:
  pgd_order: before  # cheap PGD warmup before BaB
  pgd_restarts: 30
solver:
  alpha-crown:
    iteration: 100
  beta-crown:
    iteration: 50
bab:
  timeout: {TIMEOUT}
  branching:
    method: kfsb
    candidates: 3
"""
ABCROWN_CFG.write_text(abcrown_cfg_text, encoding="utf-8")
print(f"Wrote α,β-CROWN config: {ABCROWN_CFG}")

# Build the command
cmd = [
    sys.executable,
    str(ABCROWN_DIR / "abcrown.py"),
    "--config", str(ABCROWN_CFG),
]
env = os.environ.copy()
env["PYTHONPATH"] = str(ABCROWN_DIR) + os.pathsep + env.get("PYTHONPATH", "")
print("Command:", " ".join(cmd))

t0 = time.time()
with open(ABCROWN_LOG, "w", encoding="utf-8") as logf:
    proc = subprocess.run(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        env=env,
        cwd=str(ABCROWN_DIR),
    )
wall = time.time() - t0
print(f"α,β-CROWN exit code: {proc.returncode}  wall_time: {wall:.1f}s")
print(f"Log: {ABCROWN_LOG}")

# Tail of the log for sanity
tail = ABCROWN_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]
print("\n".join(tail))

Wrote α,β-CROWN config: abcrown_work/abcrown_mnist_mlp.yaml
Command: /usr/bin/python3 /content/alpha-beta-CROWN/complete_verifier/abcrown.py --config abcrown_work/abcrown_mnist_mlp.yaml
α,β-CROWN exit code: 1  wall_time: 4.5s
Log: abcrown_work/abcrown_stdout.log
/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1480: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Traceback (most recent call last):
  File "/content/alpha-beta-CROWN/complete_verifier/abcrown.py", line 27, in <module>
    from jit_precompile import precompile_jit_kernels
  File "/content/alpha-beta-CROWN/complete_verifier/jit_precompile.py", line 21, in <module>
    from beta_CROWN_solver import LiRPANet
  File "/content/alpha-beta-CROWN/complete_verifier/beta_CROWN_solver.py", line 31, in <module>
    from attack import attack
  File "/content/alpha-beta-CROWN/complete_verifier/attack/__init__.py", line 16, in <module>
    from . impor

## 8 — Parse results into our CSV schema

In [9]:
def _normalise_status(s: str) -> str:
    s = (s or "").strip().lower()
    if s in {"safe", "unsat", "verified"}:
        return "VERIFIED"
    if s.startswith("unsafe") or s in {"sat", "falsified"}:
        return "FALSIFIED"
    if s in {"timeout", "unknown_timeout"}:
        return "TIMEOUT"
    if s in {"unknown", ""}:
        return "TIMEOUT"  # treat "unknown" as a soft timeout for cross-method comparison
    return "ERROR"


rows_out = []
raw = pd.read_csv(ABCROWN_OUT, header=None) if ABCROWN_OUT.exists() else None
if raw is None or raw.empty:
    raise RuntimeError(
        f"α,β-CROWN result file missing or empty: {ABCROWN_OUT}.\n"
        f"Inspect the log: {ABCROWN_LOG}"
    )

# α,β-CROWN's results CSV columns vary slightly across versions.
# Recent versions write: <vnnlib_path>, <status>, <time>
# Older versions: <vnnlib_path>, <onnx_path>, <status>, <time>
# Detect by column count.
if raw.shape[1] == 3:
    raw.columns = ["vnnlib", "status", "time"]
elif raw.shape[1] == 4:
    raw.columns = ["vnnlib", "onnx", "status", "time"]
else:
    raise RuntimeError(f"Unexpected α,β-CROWN result CSV shape: {raw.shape}\n{raw.head()}")

# Map vnnlib path → sample_idx via the filename pattern sample_NNNNN.vnnlib
import re
_re = re.compile(r"sample_(\d+)\.vnnlib$")
for _, r in raw.iterrows():
    p = str(r["vnnlib"])
    m = _re.search(p)
    if not m:
        continue
    sample_idx = int(m.group(1))
    rows_out.append({
        "sample_idx": sample_idx,
        "true_label": labels_by_idx.get(sample_idx, -1),
        "status":     _normalise_status(str(r["status"])),
        "time":       float(r["time"]) if pd.notna(r["time"]) else None,
        "raw_status": str(r["status"]).strip(),
    })

df_ab = pd.DataFrame(rows_out).sort_values("sample_idx").reset_index(drop=True)
Path("results").mkdir(parents=True, exist_ok=True)
out_csv = f"results/abcrown_{meta.run_name}.csv"
df_ab.to_csv(out_csv, index=False)
print(f"Wrote {out_csv}  ({len(df_ab)} rows)")
print("\nStatus counts:")
print(df_ab["status"].value_counts().to_string())
print("\nFirst 10 rows:")
print(df_ab.head(10).to_string(index=False))

RuntimeError: α,β-CROWN result file missing or empty: abcrown_work/abcrown_out.csv.
Inspect the log: abcrown_work/abcrown_stdout.log

## 9 — Summary

In [ ]:
n = len(df_ab)
n_ver  = int((df_ab["status"] == "VERIFIED").sum())
n_fal  = int((df_ab["status"] == "FALSIFIED").sum())
n_to   = int((df_ab["status"] == "TIMEOUT").sum())
n_err  = int((df_ab["status"] == "ERROR").sum())
mean_t = float(df_ab["time"].mean()) if df_ab["time"].notna().any() else float("nan")
print(f"α,β-CROWN on {meta.run_name!r} @ eps={EPS}, timeout={TIMEOUT}s, n={n}")
print(f"  VERIFIED:  {n_ver}/{n}  ({n_ver/n:.1%})")
print(f"  FALSIFIED: {n_fal}/{n}  ({n_fal/n:.1%})")
print(f"  TIMEOUT:   {n_to}/{n}  ({n_to/n:.1%})")
print(f"  ERROR:     {n_err}/{n}  ({n_err/n:.1%})")
print(f"  mean time: {mean_t:.2f}s")

In [ ]:
# Self-contained ONNX export recovery                                                                                                    !pip install -q onnx onnxruntime onnxscript
from google.colab import drive
drive.mount("/content/drive")
import os                                                                                                                                
os.chdir("/content/drive/MyDrive/thesis-formal-verification")
print("cwd:", os.getcwd())                                                                                                             
!ls runs/ 
                                                                                                                                     
import os, json
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import onnxruntime as ort
import torch
from torch import nn

# Drive should already be mounted; just make sure cwd is right
os.chdir("/content/drive/MyDrive/thesis-formal-verification")
print("cwd:", os.getcwd())

# Pick the checkpoint matching whichever pass you intend to do
CKPT_PATH = "runs/mlp_mnist/model.pt"           # <-- standard pass
# CKPT_PATH = "runs/mlp_ibp_trained/model.pt"   # <-- IBP-trained pass
WORK_DIR = Path("abcrown_work"); WORK_DIR.mkdir(parents=True, exist_ok=True)
EPS = 0.03

class MnistMlp(nn.Module):
    def __init__(self, in_dim=784, h1=128, h2=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h1); self.fc2 = nn.Linear(h1, h2); self.fc3 = nn.Linear(h2, num_classes)
        self.relu = nn.ReLU()
    def forward(self, x):
        if x.ndim == 4: x = x.view(x.shape[0], -1)
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))
    def linear_layers(self): return [self.fc1, self.fc2, self.fc3]

class _MlpFlatOnly(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.fc1, self.fc2, self.fc3, self.relu = m.fc1, m.fc2, m.fc3, m.relu
    def forward(self, x):
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))

@dataclass(frozen=True)
class CheckpointMeta:
    model_type: str
    model_kwargs: dict
    run_name: str

payload = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
raw = payload["meta"]
meta = CheckpointMeta(
    model_type=str(raw["model_type"]),
    model_kwargs=dict(raw.get("model_kwargs", {})),
    run_name=str(raw.get("run_name", "run")),
)
model = MnistMlp(**meta.model_kwargs); model.load_state_dict(payload["model_state_dict"]); model.eval()
print(f"loaded checkpoint: {meta.run_name}")

# Export
ONNX_PATH = WORK_DIR / f"{meta.run_name}.onnx"
flat = _MlpFlatOnly(model).eval()
dummy = torch.zeros(1, 784, dtype=torch.float32)
torch.onnx.export(
    flat, dummy, str(ONNX_PATH),
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=12, do_constant_folding=True, dynamo=False,
)
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
rng = np.random.RandomState(1234)
x_np = rng.uniform(0.0, 1.0, size=(100, 784)).astype(np.float32)
with torch.no_grad():
    torch_out = flat(torch.from_numpy(x_np)).cpu().numpy()
onnx_out = sess.run(["logits"], {"input": x_np})[0]
diff = float(np.max(np.abs(torch_out - onnx_out)))
print(f"ONNX exported: {ONNX_PATH}  parity max|Δ|={diff:.3e}")
assert diff < 1e-5, f"parity FAILED: {diff:.3e}"
print("OK — torch ≡ onnx within 1e-05")


Mounted at /content/drive
cwd: /content/drive/MyDrive/thesis-formal-verification
mlp_ibp_trained  mlp_mnist


ModuleNotFoundError: No module named 'onnxruntime'

In [ ]:
# === Combined: install α,β-CROWN + generate VNNLIB + run + parse ===
import os, sys, json, subprocess, time, re                                                                                               
from pathlib import Path
                                                                                                                                        
# 1. Install α,β-CROWN (idempotent; skips if already installed/cloned)
!pip install -q auto_LiRPA sortedcontainers appdirs ml_collections termcolor jsonpatch
ABCROWN_REPO = Path("/content/alpha-beta-CROWN")
if not ABCROWN_REPO.exists():
    !git clone --depth 1 https://github.com/Verified-Intelligence/alpha-beta-CROWN.git {ABCROWN_REPO}
ABCROWN_DIR = ABCROWN_REPO / "complete_verifier"
assert (ABCROWN_DIR / "abcrown.py").exists(), "abcrown.py not found after clone"
print("abcrown.py:", ABCROWN_DIR / "abcrown.py")

# 2. Imports + state assertions (kernel must still hold model/meta/WORK_DIR)
import numpy as np
import pandas as pd
import torch
from torchvision import datasets, transforms
assert "model" in dir() and "meta" in dir() and "WORK_DIR" in dir(), \
    "kernel state lost — re-run the ONNX export cell first"

# 3. Config
SUBSET_PATH = "assets/splits/mnist_eval_1000.json"
MAX_SAMPLES = 100
EPS         = 0.03
TIMEOUT     = 60
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

indices = json.loads(Path(SUBSET_PATH).read_text())["indices"][:MAX_SAMPLES]
test_ds = datasets.MNIST(root="data", train=False, download=True, transform=transforms.ToTensor())

ONNX_PATH = WORK_DIR / f"{meta.run_name}.onnx"
assert ONNX_PATH.exists(), f"missing {ONNX_PATH} — re-run ONNX export"

# 4. VNNLIB writer
def write_vnnlib(path, x0, eps, y, n_in=784, n_out=10):
    lo = np.clip(x0 - eps, 0, 1).astype(np.float64)
    hi = np.clip(x0 + eps, 0, 1).astype(np.float64)
    lines  = [f"(declare-const X_{i} Real)" for i in range(n_in)]
    lines += [f"(declare-const Y_{c} Real)" for c in range(n_out)]
    for i in range(n_in):
        lines.append(f"(assert (>= X_{i} {lo[i]:.10f}))")
        lines.append(f"(assert (<= X_{i} {hi[i]:.10f}))")
    or_clauses = " ".join(f"(>= Y_{c} Y_{y})" for c in range(n_out) if c != y)
    lines.append(f"(assert (or {or_clauses}))")
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")

specs_dir = WORK_DIR / "specs"; specs_dir.mkdir(parents=True, exist_ok=True)
labels_by_idx, csv_rows = {}, []
for i in indices:
    x_t, y_t = test_ds[int(i)]
    x0 = x_t.detach().view(-1).numpy().astype(np.float32)
    y = int(y_t); labels_by_idx[int(i)] = y
    p = specs_dir / f"sample_{int(i):05d}.vnnlib"
    write_vnnlib(p, x0, EPS, y)
    csv_rows.append((ONNX_PATH.name, f"specs/{p.name}", TIMEOUT))
(WORK_DIR / "instances.csv").write_text(
    "\n".join(f"{a},{b},{c}" for a,b,c in csv_rows) + "\n", encoding="ascii")
print(f"wrote {len(csv_rows)} VNNLIB specs")

# 5. α,β-CROWN config + run
ABCROWN_OUT = WORK_DIR / "abcrown_out.csv"
ABCROWN_LOG = WORK_DIR / "abcrown_stdout.log"
ABCROWN_CFG = WORK_DIR / "abcrown_mnist_mlp.yaml"
ABCROWN_CFG.write_text(f"""\
general:
device: {DEVICE}
csv_name: instances.csv
results_file: abcrown_out.csv
root_path: {WORK_DIR.resolve()}
conv_mode: matrix
enable_incomplete_verification: true
model:
onnx_path: null
specification:
norm: .inf
type: lp
vnnlib_path: null
attack:
pgd_order: before
pgd_restarts: 30
solver:
alpha-crown:
    iteration: 100
beta-crown:
    iteration: 50
bab:
timeout: {TIMEOUT}
branching:
    method: kfsb
    candidates: 3
""", encoding="utf-8")

cmd = [sys.executable, str(ABCROWN_DIR / "abcrown.py"), "--config", str(ABCROWN_CFG)]
env = os.environ.copy()
env["PYTHONPATH"] = str(ABCROWN_DIR) + os.pathsep + env.get("PYTHONPATH", "")
print("running:", " ".join(cmd))
t0 = time.time()
with open(ABCROWN_LOG, "w", encoding="utf-8") as f:
    proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=str(ABCROWN_DIR))
print(f"abcrown exit={proc.returncode}  wall={time.time()-t0:.1f}s")
print("--- log tail ---")
for line in ABCROWN_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-30:]:
    print(line)

# 6. Parse to results CSV
def _norm(s):
    s = (s or "").strip().lower()
    if s in {"safe","unsat","verified"}: return "VERIFIED"
    if s.startswith("unsafe") or s in {"sat","falsified"}: return "FALSIFIED"
    if s in {"timeout","unknown_timeout","unknown",""}: return "TIMEOUT"
    return "ERROR"

raw = pd.read_csv(ABCROWN_OUT, header=None) if ABCROWN_OUT.exists() else None
if raw is None or raw.empty:
    raise RuntimeError(f"no abcrown output at {ABCROWN_OUT} — see {ABCROWN_LOG}")
if raw.shape[1] == 3:    raw.columns = ["vnnlib","status","time"]
elif raw.shape[1] == 4:  raw.columns = ["vnnlib","onnx","status","time"]
else:                    raise RuntimeError(f"unexpected abcrown CSV shape: {raw.shape}")

rgx = re.compile(r"sample_(\d+)\.vnnlib$")
rows_out = []
for _, r in raw.iterrows():
    m = rgx.search(str(r["vnnlib"]))
    if not m: continue
    idx = int(m.group(1))
    rows_out.append({
        "sample_idx": idx,
        "true_label": labels_by_idx.get(idx, -1),
        "status":     _norm(str(r["status"])),
        "time":       float(r["time"]) if pd.notna(r["time"]) else None,
        "raw_status": str(r["status"]).strip(),
    })
df_ab = pd.DataFrame(rows_out).sort_values("sample_idx").reset_index(drop=True)
Path("results").mkdir(parents=True, exist_ok=True)
out_csv = f"results/abcrown_{meta.run_name}.csv"
df_ab.to_csv(out_csv, index=False)
print(f"\nwrote {out_csv} ({len(df_ab)} rows)")
print(df_ab["status"].value_counts().to_string())

ERROR: Cannot install auto-lirpa==0.2 and auto-lirpa==0.3 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
Cloning into '/content/alpha-beta-CROWN'...
remote: Enumerating objects: 560, done.
remote: Counting objects: 100% (560/560), done.
remote: Compressing objects: 100% (334/334), done.
remote: Total 560 (delta 229), reused 473 (delta 205), pack-reused 0 (from 0)
Receiving objects: 100% (560/560), 80.69 MiB | 16.50 MiB/s, done.
Resolving deltas: 100% (229/229), done.
abcrown.py: /content/alpha-beta-CROWN/complete_verifier/abcrown.py


AssertionError: kernel state lost — re-run the ONNX export cell first